# 02 · Fine-Tuning Pipeline

End-to-end LoRA fine-tuning of `Qwen 2.5-3B Instruct` (4-bit) for Text-to-SQL on Apple Silicon.

**Pipeline:**
1. Generate `lora_config.yaml` (or use the committed one)
2. Run `mlx_lm.lora` training — ~33 minutes on MacBook Pro M4 (16GB)
3. Fuse LoRA adapters into a standalone model
4. Smoke-test the fused model on a sample prompt

**Prerequisites:** Run `01_data_prep.ipynb` first to generate `data/train.jsonl` and `data/valid.jsonl`.


## Step 1 · Build LoRA training config

Writes `lora_config.yaml` consumed by `mlx_lm.lora`. Key choices:
- **16 LoRA layers** at learning rate `1e-5` — conservative, avoids overfitting on 5K samples
- **1000 iterations**, batch size 1, `max_seq_length=1024` — fits in 16GB unified memory
- **Eval every 100 steps**, checkpoint every 200 steps


In [ ]:
import yaml

MODEL_NAME = "mlx-community/Qwen2.5-3B-Instruct-4bit"
ADAPTER_DIR = "adapters"
FUSED_DIR = "fused_model"

lora_config = {
    "model": MODEL_NAME,
    "train": True,
    "data": str(DATA_DIR),
    "seed": 42,
    "lora_layers": 16,
    "batch_size": 1,
    "iters": 1000,
    "val_batches": 25,
    "learning_rate": 1e-5,
    "steps_per_report": 50,
    "steps_per_eval": 100,
    "adapter_path": ADAPTER_DIR,
    "save_every": 200,
    "max_seq_length": 1024,
}

config_path = "lora_config.yaml"
with open(config_path, "w") as f:
    yaml.dump(lora_config, f)

print(f"Config saved to {config_path}")
print(f"Model: {MODEL_NAME}")
print(f"Iterations: {lora_config['iters']}")


## Step 2 · Run LoRA fine-tuning

⏱️ **~33 minutes on M4** (16GB unified memory).

MLX streams training/validation loss to stdout — watch for divergence between train and val loss as an early overfit signal.


In [ ]:
import subprocess

print("Starting fine-tuning...")
result = subprocess.run(
    ["python", "-m", "mlx_lm.lora", "--config", config_path],
    capture_output=False, text=True
)
print("Fine-tuning complete!")


## Step 3 · Fuse adapters into a standalone model

`mlx_lm.fuse` merges the LoRA adapter weights back into the base model, producing a self-contained model in `fused_model/` ready for inference (and for the Gradio demo / Ollama deployment).


In [ ]:
import subprocess

MODEL_NAME = "mlx-community/Qwen2.5-3B-Instruct-4bit"
ADAPTER_DIR = "adapters"
FUSED_DIR = "fused_model"

subprocess.run([
    "python", "-m", "mlx_lm.fuse",
    "--model", MODEL_NAME,
    "--adapter-path", ADAPTER_DIR,
    "--save-path", FUSED_DIR
], capture_output=False, text=True)

print(f"Fused model saved to {FUSED_DIR}/")


## Step 4 · Smoke test

Quick sanity-check the fused model on a sample query before running the full evaluation suite (`03_evaluation.ipynb`).


In [ ]:
from mlx_lm import load, generate

SYSTEM_PROMPT = """You are a SQL expert. Given a database schema and a natural language question, generate the correct SQL query and a brief explanation.

Respond in this exact format:
SQL: <your sql query>
Explanation: <brief explanation>"""

FUSED_DIR = "fused_model"
model, tokenizer = load(FUSED_DIR)

test_prompt = """Database Schema:
CREATE TABLE employees (id INT, name VARCHAR(100), department VARCHAR(50), salary DECIMAL(10,2));
CREATE TABLE departments (id INT, name VARCHAR(50), budget DECIMAL(12,2));

Question: What is the average salary by department?"""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": test_prompt}
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
response = generate(model, tokenizer, prompt=prompt, max_tokens=256)
print("--- Fine-tuned Model Output ---")
print(response)

# Free memory
del model, tokenizer
